# 32B The panel as a database, and the view the model reads
Two tables from your Lab 3B panel, and one view, `holdings_v`, that you design for the model.

In [ ]:
import sys, json, pathlib, tempfile
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from askpanel import audit, catalog, clock, db, econ, gate, llm, nodes, questions, standin, tools
P = yaml.safe_load(open(ROOT / "params.yaml"))
PANEL = db.load_panel(ROOT / P["data"]["panel"])
TMP = pathlib.Path(tempfile.mkdtemp())
BASE = db.build_db(PANEL, TMP / "base.sqlite")          # tables only: works before you write the view
print(f"{len(PANEL)} panel rows, {PANEL['adsh'].nunique()} filings; scratch database {BASE.name}")

In [ ]:
con = gate.connect_ro(BASE)
for t in ["filings", "holdings"]:
    r = gate.run_query(con, f"SELECT COUNT(*) FROM {t}")
    print(t, r["rows"][0][0], "rows")
r = gate.run_query(con, "SELECT period_role, COUNT(*) FROM holdings GROUP BY period_role")
print(r["rows"], "<- two schedules per filing: the view keeps the current one")

In [ ]:
print("holdings_v must have exactly these columns:")
for c in db.VIEW_COLUMNS:
    print(f"  {c:16s} {tools.DESCRIPTIONS.get(c, '')}")

## What the check catches
A view that forgets the `period_role` filter doubles every holding. The check compares row counts and fair value per filing against the panel:

In [ ]:
wrong = """CREATE VIEW holdings_v AS
SELECT h.adsh, f.name AS bdc, f.form, f.period, h.identifier, h.investment_type, h.industry,
       h.fair_value, h.cost, h.principal, NULL AS price, h.pik_rate, h.interest_rate,
       h.spread, h.maturity, 0 AS is_debt
FROM holdings h JOIN filings f ON f.adsh = h.adsh;"""
for p in db.check_view(db.build_db(PANEL, TMP / "wrong.sqlite", wrong), PANEL):
    print("  ", p)

**Your turn (Part 0, 10 points).**
1. Write `sql/holdings_v.sql`, starting with `DROP VIEW IF EXISTS holdings_v;`: one row per **current** holding, the BDC name joined **on `adsh`**, the sixteen columns above, `price` = 100 x fair value / principal where principal > 0 and NULL otherwise (if you round, keep at least 2 decimals), `is_debt` = 1 where principal > 0 and 0 otherwise.
2. `python tools/build_db.py` prints `holdings_v: meets the contract` on the fixture.
3. Copy your Lab 3B real panel to `data/real/panel_2p.parquet` (or `.csv`), set `data.use_real: true` in `params.yaml`, and run `python tools/build_db.py` again: the same check must pass on real data.